# K-Means Clustering - Hands-On Exercises

## Scenario: SPG Visit Planning for Indonesian FMCG Retail Outlets

You are a supervisor managing SPG (Sales Promotion Girls/Guys) who visit retail outlets in Denpasar, Bali. You have data on 500 outlets and want to segment them to optimize visit planning.

**Goal:** Use K-Means clustering to identify outlet segments and create actionable visit strategies.

## Exercise 1: Load and Explore Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Load the dataset
def load_data(filepath='datasets/spg_outlets.csv'):
    data = []
    with open(filepath, 'r') as f:
        headers = f.readline().strip().split(',')
        for line in f:
            data.append(line.strip().split(','))
    
    # Extract features
    features = []
    for row in data:
        features.append([
            float(row[4]),  # monthly_sales_idr
            float(row[5]),  # sku_count
            float(row[6]),  # visit_freq_per_month
            float(row[7]),  # oos_rate
            float(row[8])   # avg_basket_idr
        ])
    return np.array(features)

X = load_data()
print(f"Loaded {len(X)} outlets with {X.shape[1]} features")
print(f"\nFeature statistics:")
print(f"  Monthly sales (IDR): mean={X[:, 0].mean():,.0f}, std={X[:, 0].std():,.0f}")
print(f"  SKU count: mean={X[:, 1].mean():.1f}, std={X[:, 1].std():.1f}")
print(f"  Visit freq/month: mean={X[:, 2].mean():.2f}, std={X[:, 2].std():.2f}")
print(f"  OOS rate: mean={X[:, 3].mean():.3f}, std={X[:, 3].std():.3f}")
print(f"  Avg basket (IDR): mean={X[:, 4].mean():,.0f}, std={X[:, 4].std():,.0f}")

**Question:** Why is feature scaling important for K-Means? Look at the range of values above.

## Exercise 2: Scale Features

K-Means uses Euclidean distance, so features with larger ranges dominate. Always scale!

In [ ]:
# TODO: Scale features to zero mean and unit variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("After scaling:")
print(f"  Mean: {X_scaled.mean(axis=0)}")
print(f"  Std: {X_scaled.std(axis=0)}")
print("\n✓ All features now have mean≈0 and std≈1")

## Exercise 3: Elbow Method to Find Optimal K

The elbow method helps us choose K by plotting inertia vs. number of clusters.

In [ ]:
# TODO: Compute inertia for k from 2 to 10
k_range = range(2, 11)
inertias = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    print(f"k={k}: inertia={kmeans.inertia_:.2f}")

# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=10)
plt.xlabel('Number of clusters (k)', fontsize=12)
plt.ylabel('Inertia', fontsize=12)
plt.title('Elbow Method For Optimal k', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print("\nLook for the 'elbow' where the curve bends.")

**Question:** Based on the elbow plot, what value of K would you choose? Why?

## Exercise 4: Silhouette Score Analysis

Silhouette score measures how well-defined clusters are. Ranges from -1 to 1 (higher is better).

In [ ]:
# TODO: Compute silhouette scores for k from 2 to 10
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"k={k}: silhouette={score:.3f}")

# Plot silhouette scores
plt.figure(figsize=(10, 6))
plt.plot(list(k_range), silhouette_scores, 'ro-', linewidth=2, markersize=10)
plt.xlabel('Number of clusters (k)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Analysis', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='gray', linestyle='--')
plt.show()

best_k = list(k_range)[np.argmax(silhouette_scores)]
print(f"\nBest k by silhouette score: {best_k}")

**Question:** Do the elbow method and silhouette score agree on the optimal K?

## Exercise 5: Fit K-Means and Analyze Clusters

Let's use K=4 (the true number of hidden segments in the data) and interpret the results.

In [ ]:
# Fit K-Means with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

print(f"Clustering complete!")
print(f"  Inertia: {kmeans.inertia_:.2f}")
print(f"  Iterations: {kmeans.n_iter_}")

# Cluster distribution
unique, counts = np.unique(labels, return_counts=True)
print(f"\nCluster sizes:")
for cluster_id, count in zip(unique, counts):
    print(f"  Cluster {cluster_id}: {count} outlets ({count/len(labels)*100:.1f}%)")

## Exercise 6: Interpret Cluster Centers

Transform cluster centers back to original scale for interpretation.

In [ ]:
# Get cluster centers in original scale
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)

feature_names = ['Monthly Sales (IDR)', 'SKU Count', 'Visit Freq/Month', 'OOS Rate', 'Avg Basket (IDR)']

print("Cluster Characteristics:\n")
for i, center in enumerate(centers_original):
    cluster_size = (labels == i).sum()
    print(f"Cluster {i} ({cluster_size} outlets):")
    print(f"  Monthly Sales: Rp {center[0]:,.0f}")
    print(f"  SKU Count: {center[1]:.0f}")
    print(f"  Visit Freq/Month: {center[2]:.2f}")
    print(f"  OOS Rate: {center[3]:.2%}")
    print(f"  Avg Basket: Rp {center[4]:,.0f}")
    print()

## Exercise 7: Create Visit Strategy

Based on cluster characteristics, recommend visit strategies for SPG team.

In [ ]:
# TODO: Analyze each cluster and suggest a visit strategy
print("Recommended Visit Strategies:\n")

for i, center in enumerate(centers_original):
    cluster_size = (labels == i).sum()
    sales = center[0]
    sku = center[1]
    oos = center[3]
    
    print(f"Cluster {i}:")
    
    # Your interpretation here!
    if sales > 20_000_000:
        strategy = "HIGH PRIORITY - Weekly visits with full merchandising support"
    elif sales > 8_000_000:
        strategy = "MEDIUM PRIORITY - Bi-weekly visits with regular stock checks"
    elif oos > 0.30:
        strategy = "NEEDS ATTENTION - Focus on reducing out-of-stock issues"
    else:
        strategy = "LOW PRIORITY - Monthly visits with basic support"
    
    print(f"  → {strategy}")
    print()

## Exercise 8: Visualize Clusters (Bonus)

Visualize the first two principal features.

In [ ]:
# Simple 2D visualization using first two features (sales vs SKU count)
plt.figure(figsize=(12, 8))

colors = ['red', 'blue', 'green', 'orange']
for i in range(4):
    cluster_mask = labels == i
    plt.scatter(X[cluster_mask, 0], X[cluster_mask, 1], 
                c=colors[i], label=f'Cluster {i}', alpha=0.6, s=50)

# Plot centers
for i, center in enumerate(centers_original):
    plt.scatter(center[0], center[1], c='black', marker='X', s=300, 
                edgecolors='white', linewidths=2)
    plt.text(center[0], center[1], f'C{i}', fontsize=12, ha='center', va='center', 
             color='white', fontweight='bold')

plt.xlabel('Monthly Sales (IDR)', fontsize=12)
plt.ylabel('SKU Count', fontsize=12)
plt.title('K-Means Clusters: Sales vs SKU Count', fontsize=14, fontweight='bold')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Tip: For better visualization of all 5 dimensions, use PCA to project to 2D!")

## Summary

**What you learned:**
1. ✅ Why feature scaling is critical for K-Means
2. ✅ How to use elbow method to choose optimal K
3. ✅ How to use silhouette score to validate clustering quality
4. ✅ How to interpret cluster centers in business terms
5. ✅ How to convert clustering results into actionable strategies

**Next steps:**
- Try different values of K and compare results
- Experiment with different features
- Use PCA to visualize all 5 dimensions in 2D
- Compare with hierarchical clustering or DBSCAN